In [20]:
import os
import re
import pickle
import numpy as np
import faiss
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer


In [21]:
BASE_DIR = os.path.abspath(".")

PDF_CONFIG = {
    "vol1": {
        "pdf_path": "Data/raw/NCO_2015_Vol1.pdf",
        "start": 45,
        "end": 250
    },
    "vol2": {
        "pdf_path": "Data/raw/NCO_2015_Vol2.pdf",
        "start": 368,
        "end": 1519
    }
}

OUTPUT_DIR = os.path.join(BASE_DIR, "Data", "embeddings")
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [22]:
def extract_pdf_range(pdf_path, start, end):
    reader = PdfReader(pdf_path)
    text = []
    for i in range(start, end):
        page_text = reader.pages[i].extract_text()
        if page_text:
            text.append(page_text)
    return "\n".join(text)


In [23]:
VOL1_CODE_RE = re.compile(r"(\d{4}\.\d{4})\s+(.*)")

def parse_volume1(text):
    roles = {}
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue

        match = VOL1_CODE_RE.match(line)
        if match:
            code = match.group(1)
            title = match.group(2)
            roles[code] = {
                "nco_2015": code,
                "title": title
            }
    return roles


In [24]:
def process_description(lines):
    full_text = " ".join(lines)

    designations = []
    if "such as:" in full_text:
        before, after = full_text.split("such as:", 1)
        designations = [
            d.strip()
            for d in re.split(r"[;,]", after)
            if len(d.strip()) > 2
        ]
        full_text = before

    return {
        "description": full_text,
        "designations": designations
    }


In [25]:
VOL2_CODE_RE = re.compile(r"\b(\d{4}\.\d{4})\b")

def parse_volume2(text):
    roles = {}
    current_code = None
    buffer = []

    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue

        if line.startswith("ISCO 08 Unit Group Details"):
            continue

        match = VOL2_CODE_RE.match(line)
        if match:
            if current_code and buffer:
                roles[current_code] = process_description(buffer)

            current_code = match.group(1)
            buffer = []
        else:
            if current_code:
                buffer.append(line)

    if current_code and buffer:
        roles[current_code] = process_description(buffer)

    return roles


In [26]:
def merge_roles(vol1, vol2):
    merged = []

    for code, v1 in vol1.items():
        role = {
            "nco_2015": code,
            "title": v1.get("title", ""),
            "description": "",
            "designations": []
        }

        if code in vol2:
            role["description"] = vol2[code]["description"]
            role["designations"] = vol2[code]["designations"]

        merged.append(role)

    return merged


In [27]:
def build_embedding_text(role):
    parts = [
        f"Job Title: {role['title']}",
        f"NCO 2015 Code: {role['nco_2015']}"
    ]

    if role["description"]:
        parts.append(f"Description: {role['description']}")

    if role["designations"]:
        parts.append("Also known as: " + ", ".join(role["designations"]))

    return "\n".join(parts)


In [28]:
print("Reading Volume 1...")
vol1_text = extract_pdf_range(**PDF_CONFIG["vol1"])
vol1_roles = parse_volume1(vol1_text)

print("Reading Volume 2...")
vol2_text = extract_pdf_range(**PDF_CONFIG["vol2"])
vol2_roles = parse_volume2(vol2_text)

print(" Merging volumes...")
merged_roles = merge_roles(vol1_roles, vol2_roles)

print(f" Total roles merged: {len(merged_roles)}")


Reading Volume 1...
Reading Volume 2...
 Merging volumes...
 Total roles merged: 3445


In [29]:
model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [build_embedding_text(r) for r in merged_roles]
embeddings = model.encode(texts, show_progress_bar=True)

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))


Batches: 100%|██████████| 108/108 [00:09<00:00, 11.39it/s]


In [30]:
faiss_path = os.path.join(OUTPUT_DIR, "nco2015.faiss")
meta_path = os.path.join(OUTPUT_DIR, "nco2015_metadata.pkl")

faiss.write_index(index, faiss_path)

with open(meta_path, "wb") as f:
    pickle.dump(merged_roles, f)

print("FAISS index and metadata saved successfully")


FAISS index and metadata saved successfully
